# Annexe 22.4 — Comprendre les bandits, simplement

**Chapitre 22 — Les problèmes de bandits (le RL dans sa forme la plus pure)**

Cette annexe **vulgarise** les notions clé du chapitre, avec des exemples chiffrés et quelques cellules de code à exécuter. Au programme :

1. Pourquoi un bandit est un MDP avec $|S| = 1$.
2. $Q_t(a)$ : la moyenne des récompenses (formule simple puis formule mathématique).
3. $Q(a)$ vs $Q(s,a)$ : bandit vs Q-Learning.
4. Anecdote MSN.com : les bandits contextuels (et pourquoi c'est souvent plus rentable que le Deep RL).
5. C'est quoi un test A/B ?
6. Les 4 algorithmes *value-based* résumés avec des métaphores humaines.

## 1. Pourquoi $|S| = 1$ ?

Dans un MDP, $S$ désigne l'**ensemble des états possibles** du système. Donc :

$$|S| = 1$$

signifie : **il n'y a qu'un seul état possible**.

Dans un problème de **bandit manchot**, l'agent n'a pas vraiment d'évolution d'état à gérer. À chaque tour, il est toujours dans la **même situation** :

> « Je dois choisir une action parmi plusieurs bras / machines / options. »

### Exemple simple

Tu as 3 machines : **Machine A**, **Machine B**, **Machine C**.

À chaque essai, tu choisis une machine, tu reçois une récompense, puis tu reviens **exactement à la même situation** :

```
État unique S0 :
Choisir A, B ou C
```

Donc l'ensemble des états est $S = \{S_0\}$, et comme il contient un seul élément : $|S| = 1$.

| Notation | Signification |
|---|---|
| $S$ | ensemble des états |
| $|S|$ | nombre d'états |
| $|S| = 1$ | un seul état |

C'est pour ça qu'un bandit est **plus simple** qu'un vrai MDP : pas de chemin, pas de transition complexe entre états, pas de récompense future à attribuer à une action passée (*credit assignment*). On étudie seulement :

> Dois-je **exploiter** l'action que je crois meilleure ? **ou** dois-je **explorer** une autre action pour apprendre ?

## 2. $Q_t(a)$ : la moyenne des récompenses

Dans un bandit, on veut estimer $Q_t(a)$ = **la valeur estimée de l'action $a$ au moment $t$**.

Autrement dit : *en moyenne, combien cette action $a$ m'a rapporté jusqu'à maintenant ?*

$$Q_t(a) = \frac{\text{somme des récompenses obtenues avec l'action } a}{\text{nombre de fois où } a \text{ a été choisie}}$$

Donc c'est juste une **moyenne**.

### Exemple très simple

L'action $a$ a été choisie 4 fois et a donné : `5, 3, 7, 5`.

$$Q_t(a) = \frac{5+3+7+5}{4} = \frac{20}{4} = 5$$

« Jusqu'à maintenant, l'action $a$ rapporte en moyenne 5. »

In [ ]:
recompenses_a = [5, 3, 7, 5]
Q = sum(recompenses_a) / len(recompenses_a)
print(f"Somme = {sum(recompenses_a)}, Nombre de choix = {len(recompenses_a)}")
print(f"Q_t(a) = {Q}")

### La formule mathématique complète

$$Q_t(a) = \frac{\sum_{i=1}^{t-1} R_i \cdot \mathbf{1}_{A_i = a}}{N_t(a)}$$

Elle dit **la même chose**, en version mathématique :

- $R_i$ : la récompense obtenue au tour $i$.
- $A_i$ : l'action choisie au tour $i$.
- $\mathbf{1}_{A_i = a}$ : une fonction indicatrice qui vaut **1 si l'action choisie au tour $i$ est bien $a$**, **0 sinon**.

Donc $R_i \cdot \mathbf{1}_{A_i = a}$ veut dire : *garde la récompense $R_i$ si l'action choisie était $a$, sinon ignore-la.*

### Exemple avec la fonction indicatrice

| Tour | Action choisie | Récompense | Est-ce l'action $a$ ? | Récompense gardée |
|---|---|---|---|---|
| 1 | a | 5 | 1 | 5 |
| 2 | b | 10 | 0 | 0 |
| 3 | a | 3 | 1 | 3 |
| 4 | c | 8 | 0 | 0 |
| 5 | a | 7 | 1 | 7 |

Somme gardée = $5+0+3+0+7 = 15$. L'action $a$ a été choisie 3 fois, donc $Q_t(a) = 15/3 = 5$.

In [ ]:
# Tours : (action, recompense)
tours = [("a", 5), ("b", 10), ("a", 3), ("c", 8), ("a", 7)]
a = "a"

# 1{A_i = a} appliqué à chaque tour
gardees = [r * (1 if action == a else 0) for action, r in tours]
N = sum(1 for action, _ in tours if action == a)

print("Récompenses gardées :", gardees)
print(f"Somme = {sum(gardees)}, N_t(a) = {N}")
print(f"Q_t(a) = {sum(gardees) / N}")

> **Phrase la plus simple :** $Q_t(a)$, c'est la **moyenne des récompenses** obtenues chaque fois qu'on a choisi l'action $a$ avant le moment $t$.

## 3. $Q(a)$ vs $Q(s,a)$ : est-ce la même chose que le Q-Learning ?

Oui, c'est la **même idée générale**, mais **pas exactement le même contexte**. Dans les deux cas, $Q$ veut dire :

> Quelle est la valeur estimée d'une action ? (Si je choisis cette action, combien je pense gagner ?)

**Dans un bandit**, il n'y a qu'un seul état → on écrit seulement $Q(a)$ :

```
Action A -> rapporte en moyenne 5
Action B -> rapporte en moyenne 8
Action C -> rapporte en moyenne 2
```

L'agent pense que **B est meilleure**, car $Q(B) = 8$.

**Dans le Q-Learning**, on est dans un vrai MDP avec plusieurs états → on écrit $Q(s,a)$ :

```
État : je suis en case (2,3)
Action : aller à droite
Q(s, droite) = 0.75
```

La valeur dépend de **deux choses** : l'état actuel $s$ **et** l'action choisie $a$.

| Cas | Forme de $Q$ | Signification |
|---|---|---|
| **Bandit** | $Q(a)$ | valeur d'une action |
| **Q-Learning** | $Q(s,a)$ | valeur d'une action **dans un état donné** |

> **IMPORTANT :** dans les bandits, on apprend $Q(a)$, la valeur d'une action. Dans le Q-Learning, on **généralise** cette idée à $Q(s,a)$, la valeur d'une action dans un état précis.

## 4. Anecdote MSN.com : les bandits contextuels

> La publication Microsoft Research / Yahoo! la plus connue sur la reco de news avec bandits contextuels date de **2010** (Li et al., *A Contextual-Bandit Approach to Personalized News Article Recommendation*).

### Le problème de MSN.com

Chaque visiteur arrive sur la page d'accueil. Le système doit décider : **quel article montrer à cette personne maintenant ?** (politique, sport, finance, météo, technologie...). Il choisit un article, observe si l'utilisateur **clique**, puis apprend. C'est exactement un **bandit**.

### Pourquoi « contextuel » ?

Un bandit classique choisit entre A, B, C, D. Un **bandit contextuel** ajoute du **contexte** : pays, langue, heure, type d'appareil, historique de clics, catégorie préférée, popularité récente de l'article...

Le système ne dit plus « l'article A est bon », mais :

> Pour **ce type d'utilisateur**, à **ce moment**, dans **ce contexte**, l'article A semble le meilleur choix.

### Où est le RL ?

| Élément RL | Dans MSN.com |
|---|---|
| agent | le système de recommandation |
| action | choisir quel article afficher |
| récompense | clic, lecture, engagement |
| apprentissage | améliorer les prochains choix |

Mais ce n'est **pas du Deep RL** complexe : pas besoin de planifier 50 actions dans le futur. L'objectif est direct : *je montre un article → clic ou non → j'apprends.*

### Pourquoi c'est rentable ?

Parce que la décision est répétée **des millions de fois**. Même une petite amélioration a un effet énorme :

$$+0{,}5\% \text{ de clics} \times \text{millions de décisions/jour} = \text{beaucoup plus de revenus}$$

### Pourquoi pas directement du Deep RL ?

| Deep RL | Bandit contextuel |
|---|---|
| plus complexe, plus coûteux | plus simple, plus rapide |
| difficile à stabiliser / expliquer | facile à déployer et à mesurer |
| lent à entraîner | adapté aux décisions immédiates |

> **À retenir :** le Deep RL résout des problèmes **très complexes**. Les bandits contextuels résolvent **très bien des problèmes simples mais répétés des millions de fois** — c'est là qu'est leur valeur industrielle.

## 5. C'est quoi un test A/B ?

Un test A/B compare **deux versions** d'une même chose pour voir laquelle fonctionne le mieux. Exemple :

- **Version A** : bouton bleu « Commencer »
- **Version B** : bouton orange « Télécharger gratuitement »

On montre A à une moitié des utilisateurs, B à l'autre moitié, puis on mesure :

```
Version A : 1000 visiteurs -> 80 clics  -> 8 %
Version B : 1000 visiteurs -> 120 clics -> 12 %
```

La version **B** est meilleure. L'idée : *ne pas deviner, tester avec des données réelles* (titre, bouton, image, pub, page d'inscription, email...).

### Différence avec les bandits

| Méthode | Idée |
|---|---|
| **Test A/B** | Comparer deux versions avec une répartition **fixe** (50/50), même si B devient clairement meilleur on continue jusqu'à la fin |
| **Bandit** | Tester plusieurs options, puis **favoriser progressivement la meilleure** (montrer B plus souvent dès qu'il marche mieux) |

> **Phrase simple :** un test A/B montre deux versions à deux groupes pour mesurer laquelle donne le meilleur résultat. C'est une décision basée sur les **données** plutôt que sur l'intuition. Le bandit, lui, **réoriente le trafic en temps réel** vers la version gagnante.

## 6. Les 4 algorithmes *value-based* (métaphores humaines)

Le dilemme à gérer est toujours le même :

```
Explorer  = tester pour apprendre
Exploiter = choisir ce qui semble déjà meilleur
```

| Algorithme | Idée simple | Métaphore humaine |
|---|---|---|
| **ε-greedy** | La plupart du temps je choisis la meilleure action connue ; parfois (proba $\varepsilon$) je choisis au hasard pour explorer. | Le **pragmatique** : il a ses habitudes, mais garde un peu de curiosité. |
| **Optimistic Initial Values** | Au début, je donne une très bonne note artificielle à toutes les actions ; comme elles semblent prometteuses, je suis obligé de les tester. | L'**optimiste naïf** : il croit que tout le monde est excellent jusqu'à preuve du contraire. |
| **UCB** | Je choisis l'action qui combine bonne performance connue **et** forte incertitude (bonus d'exploration calculé). | Le **manager rationnel** : « ce candidat semble bon mais je le connais peu → donnons-lui une chance. » |
| **Thompson Sampling** | Je maintiens une **croyance probabiliste** sur chaque action, puis je tire une option au hasard selon ces croyances. | Le **joueur intuitif** : il parie selon sa confiance, mais accepte l'incertitude. |

### En une ligne chacun

```
eps-greedy  -> explore parfois au hasard
Optimistic  -> force l'exploration au début
UCB         -> explore ce qui est prometteur mais incertain
Thompson    -> explore selon des probabilités de confiance
```

> **IMPORTANT :** ces algorithmes sont **différentes façons de répondre à la même question** — dois-je choisir l'action que je crois déjà meilleure, ou tester une autre action pour apprendre davantage ?

---

Pour **voir ces algorithmes en action** et comparer leurs performances, ouvrez le notebook `22.2-Revision-et-coder-Bandits.ipynb` *(dossier `Materials/`)*.